PyTorch Dataset 与 DataLoader

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.datasets import load_iris

In [12]:
samples = [
    (torch.tensor([0, 00, 000]), 0),
    (torch.tensor([1, 10, 100]), 1),
    (torch.tensor([2, 20, 200]), 2),
    (torch.tensor([3, 30, 300]), 3),
    (torch.tensor([4, 40, 400]), 4),
    (torch.tensor([5, 50, 500]), 5),
    (torch.tensor([6, 60, 600]), 6),
    (torch.tensor([7, 70, 700]), 7),
    (torch.tensor([8, 80, 800]), 8),
    (torch.tensor([9, 90, 900]), 9),
]
# DataLoader 让数据按需组成 batch：
loader = DataLoader(samples, batch_size=2, shuffle=True)
for x, y in loader:
    # [2, 3] 表示每一个批次有2个样本， 每个样本有3个特征数据
    print(x.shape, y.shape)
    print(x, y)
    print("=" * 30)


torch.Size([2, 3]) torch.Size([2])
tensor([[  4,  40, 400],
        [  8,  80, 800]]) tensor([4, 8])
torch.Size([2, 3]) torch.Size([2])
tensor([[  5,  50, 500],
        [  9,  90, 900]]) tensor([5, 9])
torch.Size([2, 3]) torch.Size([2])
tensor([[  2,  20, 200],
        [  7,  70, 700]]) tensor([2, 7])
torch.Size([2, 3]) torch.Size([2])
tensor([[  3,  30, 300],
        [  1,  10, 100]]) tensor([3, 1])
torch.Size([2, 3]) torch.Size([2])
tensor([[  0,   0,   0],
        [  6,  60, 600]]) tensor([0, 6])


In [17]:
samples = [
    (torch.tensor([[0, 00, 000, 0000], [1, 10, 100, 1000]]), 0),
    (torch.tensor([[2, 20, 200, 2000], [3, 30, 300, 3000]]), 1),
    (torch.tensor([[4, 40, 400, 4000], [5, 50, 500, 5000]]), 4),
    (torch.tensor([[6, 60, 600, 6000], [7, 70, 700, 7000]]), 7),
]
loader = DataLoader(samples, batch_size=3, shuffle=True)
for x, y in loader:
    # [3, 2, 4] 表示每一个批次有3个样本， 每个样本都是[2,4]的特征数据
    print(x.shape, y.shape)
    print(x, y)
    print("=" * 30)
# 默认 collate 会把可堆叠的样本字段合并成批次张量。上例最后一个 batch 只有一条记录。

torch.Size([3, 2, 4]) torch.Size([3])
tensor([[[   4,   40,  400, 4000],
         [   5,   50,  500, 5000]],

        [[   2,   20,  200, 2000],
         [   3,   30,  300, 3000]],

        [[   6,   60,  600, 6000],
         [   7,   70,  700, 7000]]]) tensor([4, 1, 7])
torch.Size([1, 2, 4]) torch.Size([1])
tensor([[[   0,    0,    0,    0],
         [   1,   10,  100, 1000]]]) tensor([0])


In [18]:
## 3. Dataset 与 DataLoader 的分工

# 可以把二者理解为数据读取的两个层次：【先有数据集，再由数据集转化为数据加载器】
# 数据源 -> Dataset[index] 读取/转换一条样本 -> DataLoader 取样并组合 batch -> 训练循环 -> 模型
#
# - `Dataset`：描述数据集及单个样本的读取方式。典型接口是 `__len__()` 和 `__getitem__(index)`。
# - `DataLoader`：包裹一个 Dataset，产生可迭代的 batch。控制 `batch_size`、`shuffle`、`num_workers`、`collate_fn` 等。
# - 训练循环：取出 batch，将数据放到合适设备，调用模型和损失函数。
#
# DataLoader 本身通常不负责把整个数据集永久放进显存，也不替模型做训练或自动搬运数据到 GPU。

In [32]:
# 使用 load_iris 来构造数据加载器
iris_dataset = load_iris()
features = iris_dataset.data
labels = iris_dataset.target
print(features.shape, labels.shape)

# TensorDataset 的作用是：将多个 Tensor 按照样本索引组合起来，让特征和标签一一对应。
# TensorDataset：定义一条样本由哪些 Tensor 组成。
# 所有传给 `TensorDataset` 的 Tensor 第 0 维长度必须相同。索引同一个位置会得到一条样本的 tuple：
# 分类标签通常是类别索引 `long`；回归目标通常是浮点数 float32 。Dataset 返回的数据类型应与模型和损失函数的约定一致。
src_dataset = TensorDataset(torch.tensor(features, dtype=torch.float32), torch.tensor(labels, dtype=torch.long))
x0, y0 = src_dataset[0]
print(src_dataset[0])
print("x0.shape: ", x0.shape, y0.shape, y0)

# DataLoader 的作用是：从数据集中按批次取出样本，方便训练模型。
# DataLoader：决定如何分批、打乱和读取样本。
# - 训练集通常打乱，让每轮 batch 组成和样本顺序不同。 - 验证集和测试集通常不需要打乱，便于结果复现和对应样本。
loader = DataLoader(src_dataset, batch_size=16, shuffle=True) # ,drop_last=True 表示是否丢弃最后一个批次的数据
for x, y in loader:
    # for x, y in loader：每次获取一个批次的数据，供模型训练或推理使用。
    print(x.shape, y.shape)


# DataLoader 主要优点：
# 1. 控制内存使用：不必每次都将全部数据送入模型。
# 2. 方便梯度更新：每批数据计算一次损失并更新模型参数。
# 3. 随机打乱样本：减少模型总是按照固定顺序读取样本带来的影响。
# 4. 统一训练代码：数据集变大后，依然可以使用相同的批次读取方式。

(150, 4) (150,)
(tensor([5.1000, 3.5000, 1.4000, 0.2000]), tensor(0))
x0.shape:  torch.Size([4]) torch.Size([]) tensor(0)
torch.Size([16, 4]) torch.Size([16])
torch.Size([16, 4]) torch.Size([16])
torch.Size([16, 4]) torch.Size([16])
torch.Size([16, 4]) torch.Size([16])
torch.Size([16, 4]) torch.Size([16])
torch.Size([16, 4]) torch.Size([16])
torch.Size([16, 4]) torch.Size([16])
torch.Size([16, 4]) torch.Size([16])
torch.Size([16, 4]) torch.Size([16])
torch.Size([6, 4]) torch.Size([6])


In [29]:
## 5. 创建 DataLoader：常用参数
#
# loader = DataLoader(
#     dataset,
#     batch_size=32,
#     shuffle=True,
#     num_workers=0,
#     drop_last=False,
# )
#
# | 参数 | 作用 | 常见建议 |
# | --- | --- | --- |
# | `dataset` | 数据来源，通常是 Dataset | 先确认单条样本返回内容 |
# | `batch_size` | 每个 batch 的样本数 | 从显存/内存能承受的值开始 |
# | `shuffle` | 每轮迭代前打乱索引 | 训练集通常 `True`；验证/测试通常 `False` |
# | `num_workers` | 并行加载 worker 进程数 | 初始用 `0` 排错，再按环境调优 |
# | `drop_last` | 是否丢弃最后一个不完整 batch | 训练可按需要开启；评估通常关闭 |
# | `collate_fn` | 将若干样本合成 batch 的函数 | 默认规则不适合变长/复杂样本时自定义 |
# | `pin_memory` | 将 CPU batch 放入页锁定内存 | CUDA 训练可尝试配合异步拷贝 |
# | `persistent_workers` | epoch 间保留 worker 进程 | 仅 `num_workers > 0` 时有效，适合多轮训练 |
# | `prefetch_factor` | 每个 worker 预取 batch 数 | worker 开启时可调，增加 RAM 占用 |
# | `generator` | 控制 sampler 的随机数生成器 | 需要可复现的 shuffle 时使用 |
#
# `batch_size` 通常不能与自定义 `batch_sampler` 一起指定。`shuffle` 与 `sampler` 也不能同时使用，因为 sampler 已经决定索引顺序。

In [67]:
# 自定义实现一个ToyDataset， 必须实现以下两个函数
# - `__len__()`：返回数据集样本总数。
# - `__getitem__(index)`：返回一条样本，通常是 Tensor、tuple、list 或 dict。
class ToyDataset(Dataset):
    def __init__(self, count=100):
        self.features = torch.randn(count, 4, dtype=torch.float32)
        # 模拟使用每一个样本的第一个特征数据是否大于0来作为目标标签
        self.labels = (self.features[:, 0] > 0).long()
    def __len__(self) -> int:
        return len(self.features)
    def __getitem__(self, index: int):
        # 这个函数将支持ToyDataset对象下标获取数据
        # 常见的返回方式，1直接返回元组，或返回字典
        # return feature_tensor, label_tensor
        # return {"image": image, "label": label, "path": str(path)}
        return self.features[index], self.labels[index]

toy_dataset = ToyDataset(100)
print(toy_dataset[0])
loader = DataLoader(toy_dataset, batch_size=4, shuffle=True)


(tensor([-1.6127,  0.4232, -0.4340,  0.8178]), tensor(0))


In [68]:
# ## 8. 划分训练集、验证集与测试集
from torch.utils.data import random_split
toy_dataset = ToyDataset(100)
# train:valid:test = 8:1:1
train_size = 80
valid_size = 10
test_size = len(toy_dataset) - train_size - valid_size
generator = torch.Generator().manual_seed(23)
# `random_split` 返回 `Subset`，包含原始 Dataset 和对应索引，不会把每条样本复制一份。分层分类、按时间切分、按用户/病人分组切分时，不应盲目随机切分；要按任务防止类别比例失衡、时间穿越或同一主体泄漏。
train_set, valid_set, test_set = random_split(toy_dataset, [train_size, valid_size, test_size], generator=generator)
train_loader = DataLoader(train_set, batch_size=4, shuffle=True)

In [69]:
### 8.1 先切分，再拟合预处理
# 标准化均值/方差、缺失值填充值、词表等统计量需要从训练集拟合，再复用于验证和测试集。先对全量数据算统计量再划分会泄漏验证/测试信息。
# 即使用train来进行数据训练fit，之后valid/test要使用的话再使用transport进行转化

In [75]:
## 9. 把 DataLoader 接入训练循环
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 输入四个特征，输出一个数字标签的二分类任务
model = torch.nn.Linear(4, 1)
model.to(device)

epochs = 20
criterion = torch.nn.BCEWithLogitsLoss() # 分类损失 BCEWithLogitsLoss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 开启训练模式
model.train()
for epoch in range(epochs):
    # 记录样本数和总的损失数
    total_samples = 0
    total_loss = 0
    # 训练每一个数据批次
    for x, y in train_loader:
        # 先将数据放到对应的设备上
        # 一个 batch 通常是 CPU Tensor；模型和输入必须位于兼容设备。常用写法是在训练循环中把每个 batch 移至设备，而不是将整个 Dataset 放进 GPU。
        # 因为是二分类，loss使用了BCEWithLogitsLoss()，而BCEWithLogitsLoss()对应的标签类型是float32，因此这里必须对现有数据进行一次数据类型转化
        x, y = x.to(device), y.float().to(device).unsqueeze(1)
        # 清空梯度
        optimizer.zero_grad(set_to_none=True)
        # 前向传播
        logits = model(x)
        # 计算损失
        loss = criterion(logits, y)
        # 计算梯度，反向传播
        loss.backward()
        # 更新参数
        optimizer.step()
        # 累计损失
        total_samples += x.size(0)
        total_loss += loss.detach().item() * x.size(0)

    train_loss = total_loss / total_samples

    if epoch % 1 == 0:
        print(epoch, total_loss, total_samples, train_loss)

model.eval()
test_loader = DataLoader(test_set, batch_size=4, shuffle=False)
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.float().to(device).unsqueeze(1)
        logits = model(x)
        loss = criterion(logits, y)
        preds = (logits[:, 0] > 0).long()
        print("预测结果：", y, preds)


0 72.52124381065369 80 0.9065155476331711
1 71.54409265518188 80 0.8943011581897735
2 70.63316059112549 80 0.8829145073890686
3 69.6852662563324 80 0.871065828204155
4 68.79860019683838 80 0.8599825024604797
5 67.87609434127808 80 0.8484511792659759
6 67.04308271408081 80 0.8380385339260101
7 66.17411661148071 80 0.8271764576435089
8 65.3242301940918 80 0.8165528774261475
9 64.53100776672363 80 0.8066375970840454
10 63.694175481796265 80 0.7961771935224533
11 62.93596124649048 80 0.7866995155811309
12 62.14491891860962 80 0.7768114864826202
13 61.390883922576904 80 0.7673860490322113
14 60.63313412666321 80 0.7579141765832901
15 59.8843469619751 80 0.7485543370246888
16 59.18104600906372 80 0.7397630751132965
17 58.49418830871582 80 0.7311773538589478
18 57.77128624916077 80 0.7221410781145096
19 57.10690522193909 80 0.7138363152742386
预测结果： tensor([[1.],
        [0.],
        [0.],
        [1.]]) tensor([1, 1, 1, 1])
预测结果： tensor([[0.],
        [0.],
        [0.],
        [0.]]) tenso